# EEGMamba: 10-epoch fine-tuning on a single subject-disjoint split
Use all 123 subjects across train, validation, and test; all 30 seconds of each trial in three ten-second windows. PyTorch Mamba2 fallback; native CUDA parity unverified.


In [ ]:
import os,sys,json,random,time,ast,hashlib
from pathlib import Path
import numpy as np,pandas as pd,torch
from torch import nn
from scipy.signal import resample_poly
from sklearn.metrics import balanced_accuracy_score,accuracy_score
root=Path(r'D:\EEG Experiment Artifacts\2026-11-02');source=root/'mamba_sources'/'EEGMamba';weights=root/'mamba_sources'/'EEGMamba_weights'/'pretrained_EEGMamba.pth'
sys.path[:0]=[str(root/'pretrained_experiments'/'reve_dependencies'),str(root/'pretrained_experiments'/'dependencies')]
from einops import rearrange
from transformers import Mamba2Config
from transformers.models.mamba2.modeling_mamba2 import Mamba2Mixer
seed=42;random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed);torch.set_num_threads(4)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu');assert device.type=='cuda'
protocol=json.loads((root/'binary_de_psd'/'protocol.json').read_text(encoding='utf8'));split=protocol['splits'][0]
subjects=np.repeat(np.arange(123),24);video_ids=np.r_[0:12,16:28];labels=np.tile(np.r_[np.zeros(12,dtype=np.int64),np.ones(12,dtype=np.int64)],123)
indices={k:np.flatnonzero(np.isin(subjects,split[k])) for k in ('train','val','test')}
assert not set(split['train'])&set(split['val']) and not set(split['test'])&set(split['train']+split['val'])
raw=np.load(root/'cache'/'raw_windows_30ch.npy',mmap_mode='r');assert raw.shape==(3444,30,30,250)
code=(source/'models'/'eegmamba.py').read_text(encoding='utf8');node=next(x for x in ast.parse(code).body if isinstance(x,ast.ClassDef) and x.name=='PatchEmbedding');exec(compile(ast.Module(body=[node],type_ignores=[]),str(source/'models'/'eegmamba.py'),'exec'))
config=Mamba2Config(hidden_size=200,expand=2,num_heads=8,head_dim=50,state_size=64,n_groups=1,layer_norm_epsilon=1e-5,chunk_size=32)
class Layer(nn.Module):
 def __init__(self,i):
  super().__init__();self.norm=nn.RMSNorm(200,eps=1e-5);self.mixer=Mamba2Mixer(config,layer_idx=i)
class Encoder(nn.Module):
 def __init__(self):
  super().__init__();self.layers=nn.ModuleList([Layer(i) for i in range(12)]);self.norm_f=nn.RMSNorm(200,eps=1e-5)
 def forward(self,x):
  residual=None
  for layer in self.layers:
   residual=x if residual is None else x+residual
   x=layer.mixer(layer.norm(residual));x=x.flip(1);residual=residual.flip(1)
  return self.norm_f(x+residual)
class EEGMamba(nn.Module):
 def __init__(self):
  super().__init__();self.patch_embedding=PatchEmbedding(200,200,200,30);self.encoder=Encoder();self.proj_out=nn.Sequential(nn.Linear(200,200))
 def forward(self,x):
  batch,ch,seq,_=x.shape;h=self.patch_embedding(x,mask=None);h=self.encoder(rearrange(h,'b c t d -> b (c t) d'));return rearrange(h,'b (c t) d -> b c t d',c=ch,t=seq)
model=EEGMamba();state=torch.load(weights,map_location='cpu',weights_only=True);model.load_state_dict(state,strict=True);model=model.to(device).eval()
for param in model.parameters():param.requires_grad_(False)
print('DEVICE',torch.cuda.get_device_name(0),'STRICT WEIGHT LOAD OK','PARAMETERS',sum(x.numel() for x in model.parameters()),flush=True)


In [ ]:
# Fine-tune on all three non-overlapping ten-second windows of each trial.
output=root/'mamba_experiments'/'eegmamba_finetune_single_split_10';output.mkdir(parents=True,exist_ok=True)
manifest={'model':'EEGMamba end-to-end finetuning','implementation':'author patch embedding and residual/reverse order; Transformers PyTorch Mamba2 mixer','weights_sha256':hashlib.sha256(weights.read_bytes()).hexdigest(),'protocol':'single subject-disjoint split','split':split,'seed':seed,'epochs':10,'learning_rate':1e-5,'window':'three non-overlapping 10-second windows covering each trial; cached per-trial/channel zscore, 250 to 200 Hz','pooling':'mean of learned encoder tokens','notes':'native CUDA Mamba2 unavailable; numerical parity with native implementation unverified'}
manifest_path=output/'config.json'
if manifest_path.exists():assert json.loads(manifest_path.read_text(encoding='utf8'))==manifest
else:manifest_path.write_text(json.dumps(manifest,indent=2),encoding='utf8')
for param in model.parameters():param.requires_grad_(True)
head=nn.Linear(200,2).to(device);optimizer=torch.optim.AdamW(list(model.parameters())+list(head.parameters()),lr=1e-5,weight_decay=1e-3)
rng=np.random.default_rng(seed);history=[];best=-1;start_epoch=1
last_checkpoint=output/'last_checkpoint.pt'
if last_checkpoint.exists():
 state=torch.load(last_checkpoint,map_location=device,weights_only=False);model.load_state_dict(state['backbone']);head.load_state_dict(state['head']);optimizer.load_state_dict(state['optimizer']);history=state['history'];best=state['best'];rng.bit_generator.state=state['numpy_rng'];torch.set_rng_state(state['torch_rng'].cpu());torch.cuda.set_rng_state_all([x.cpu() for x in state['cuda_rng']]);start_epoch=state['epoch']+1
 print('RESUMING FROM EPOCH',start_epoch,flush=True)
def run(name,training):
 model.train(training);head.train(training);ids=indices[name].copy()
 if training:rng.shuffle(ids)
 preds=[];ys=[];logits_all=[];losses=[]
 for j in ids:
  raw_idx=int(subjects[j])*28+int(video_ids[j%24]);signal=np.asarray(raw[raw_idx,:,:,:],dtype=np.float32).transpose(1,0,2).reshape(30,7500);signal=resample_poly(signal,4,5,axis=-1).reshape(30,3,10,200).transpose(1,0,2,3).copy();x=torch.from_numpy(signal).to(device);y=torch.full((3,),int(labels[j]),dtype=torch.long,device=device)
  with torch.set_grad_enabled(training):
   logits=head(model(x).mean(dim=(1,2)));loss=nn.functional.cross_entropy(logits,y)
   if training:optimizer.zero_grad(set_to_none=True);loss.backward();optimizer.step()
  preds.extend(logits.argmax(1).detach().cpu().numpy().tolist());ys.extend(y.cpu().numpy().tolist());logits_all.extend(logits.detach().cpu().numpy());losses.append(float(loss.detach().cpu()))
 return {'loss':float(np.mean(losses)),'accuracy':float(accuracy_score(ys,preds)),'balanced_accuracy':float(balanced_accuracy_score(ys,preds))},np.array(logits_all),np.array(preds),np.array(ys)
for epoch in range(start_epoch,11):
 t=time.perf_counter();tr,_,_,_=run('train',True);va,logits,pred,y=run('val',False);row={'epoch':epoch,'train_loss':tr['loss'],'val_loss':va['loss'],'val_accuracy':va['accuracy'],'val_balanced_accuracy':va['balanced_accuracy'],'seconds':time.perf_counter()-t};history.append(row);pd.DataFrame(history).to_csv(output/'history.csv',index=False)
 if va['balanced_accuracy']>best:
  best=va['balanced_accuracy'];torch.save({'backbone':model.state_dict(),'head':head.state_dict(),'epoch':epoch},output/'best_model.pt');np.savez_compressed(output/'validation_predictions.npz',logits=logits,predictions=pred,labels=y,subjects=np.repeat(subjects[indices['val']],3))
 state={'backbone':model.state_dict(),'head':head.state_dict(),'optimizer':optimizer.state_dict(),'epoch':epoch,'history':history,'best':best,'numpy_rng':rng.bit_generator.state,'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all()}
 tmp=output/'last_checkpoint.tmp';torch.save(state,tmp);tmp.replace(last_checkpoint)
 print('EPOCH',epoch,json.dumps(row),flush=True)
checkpoint=torch.load(output/'best_model.pt',map_location=device,weights_only=True);model.load_state_dict(checkpoint['backbone']);head.load_state_dict(checkpoint['head']);test,logits,pred,y=run('test',False);np.savez_compressed(output/'test_predictions.npz',logits=logits,predictions=pred,labels=y,subjects=np.repeat(subjects[indices['test']],3))
result={'best_validation_balanced_accuracy':best,'best_epoch':checkpoint['epoch'],'epochs':10,'test':test,'train_subjects':len(split['train']),'validation_subjects':len(split['val']),'test_subjects':len(split['test'])}
(output/'metrics.json').write_text(json.dumps(result,indent=2),encoding='utf8');print('COMPLETE',json.dumps(result),flush=True)
